# Outplay Saliency Viewport PoC

This notebook finds a fit-inside 9:16, 1:1, or 16:9 viewport for every frame in eight deterministic training clips.

It compares:

1. **Independent baseline** — the smallest candidate whose best placement has at least 90% of the strongest candidate's saliency mass.
2. **Temporal dynamic programming** — a 32-frame path that trades saliency coverage against viewport-center velocity and aspect-ratio switches.

The source saliency maps are ground truth (`all_maps`), not model predictions. The result is a review aid only; it does not modify training.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation

# Support execution from ViNet_S/, ViNet_S/notebooks/, or the repository root.
for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent, Path.cwd().resolve() / "ViNet_S"):
    if (candidate / "outplay_dataset.py").exists():
        VINET_S_DIR = candidate
        break
else:
    raise FileNotFoundError("Could not locate ViNet_S/outplay_dataset.py from the current working directory")
sys.path.insert(0, str(VINET_S_DIR))

from outplay_dataset import _load_frame, _load_map, frame_names, window_path

plt.rcParams["figure.dpi"] = 120
plt.rcParams["animation.html"] = "jshtml"

In [ ]:
# Configuration
SPLIT_ROOT = Path(
    os.environ.get(
        "OUTPLAY_SPLIT_ROOT",
        "/home/nitesh_sawadia/mnt/ml-fanpulse/sports_saliency/vinet_outplay/split_stride8",
    )
)
SPLIT = "train"
NUM_CLIPS = 8
COVERAGE_THRESHOLD = 0.90

# Temporal dynamic-programming penalties. Both are measured in normalized units.
CENTER_VELOCITY_PENALTY = 0.20
ASPECT_SWITCH_PENALTY = 0.03

ASPECT_RATIOS: Dict[str, Tuple[int, int]] = {
    "9:16": (9, 16),
    "1:1": (1, 1),
    "16:9": (16, 9),
}

if not torch.cuda.is_available():
    raise RuntimeError("Viewport convolution and temporal search require a CUDA GPU")
DEVICE = torch.device("cuda")

if not SPLIT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Outplay split root is unavailable: {SPLIT_ROOT}. "
        "Run this notebook on the VM or set OUTPLAY_SPLIT_ROOT."
    )

print(f"ViNet-S directory: {VINET_S_DIR}")
print(f"Outplay split root: {SPLIT_ROOT}")
print(f"GPU: {torch.cuda.get_device_name(DEVICE)}")
print(f"Temporal penalties: center={CENTER_VELOCITY_PENALTY}, aspect={ASPECT_SWITCH_PENALTY}")

In [ ]:
class ManifestClipSampler:
    """Lightweight notebook reader: avoids scanning all goofys windows before the PoC starts."""

    def __init__(self, split_root: Path, split: str):
        with (split_root / "split_manifest.json").open() as handle:
            manifest = json.load(handle)
        self.paths = [
            window_path(str(split_root), window)
            for window in manifest["windows"]
            if window.get("split") == split
        ]
        if not self.paths:
            raise RuntimeError(f"No {split!r} windows in {split_root}")

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        path = self.paths[index]
        frames = torch.stack([_load_frame(os.path.join(path, "images", name)) for name in frame_names()])
        maps = torch.stack([_load_map(os.path.join(path, "maps", name)) for name in frame_names()])
        return frames.float(), maps[-1].clone().float(), maps.float()


dataset = ManifestClipSampler(SPLIT_ROOT, SPLIT)
manifest_indexes = np.linspace(0, len(dataset) - 1, NUM_CLIPS, dtype=int).tolist()

sample_catalog = pd.DataFrame(
    {
        "dataset_index": manifest_indexes,
        "window_path": [dataset.paths[index] for index in manifest_indexes],
    }
)
display(sample_catalog)

# Each item returns clip [T,C,H,W], last map [H,W], and all per-frame maps [T,H,W].
clip0, last_map0, maps0 = dataset[manifest_indexes[0]]
assert clip0.shape == (32, 3, 224, 384)
assert maps0.shape == (32, 224, 384)
assert torch.equal(last_map0, maps0[-1])
print(f"Clip shape: {tuple(clip0.shape)} | saliency shape: {tuple(maps0.shape)}")

In [ ]:
@dataclass(frozen=True)
class Viewport:
    ratio: str
    height: int
    width: int

    @property
    def area(self) -> int:
        return self.height * self.width


def fit_inside_viewport(ratio_name: str, ratio: Tuple[int, int], height: int, width: int) -> Viewport:
    """Return the largest exact integer width:height viewport that fits in the frame."""
    numerator, denominator = ratio
    scale = min(height // denominator, width // numerator)
    if scale < 1:
        raise ValueError(f"{ratio_name} cannot fit inside {width}x{height}")
    return Viewport(ratio_name, denominator * scale, numerator * scale)


def make_viewports(height: int, width: int) -> List[Viewport]:
    return [fit_inside_viewport(name, ratio, height, width) for name, ratio in ASPECT_RATIOS.items()]


VIEWPORTS = make_viewports(int(maps0.shape[-2]), int(maps0.shape[-1]))
viewport_table = pd.DataFrame(
    [
        {
            "ratio": viewport.ratio,
            "height_px": viewport.height,
            "width_px": viewport.width,
            "area_px2": viewport.area,
            "area_fraction": viewport.area / (maps0.shape[-2] * maps0.shape[-1]),
        }
        for viewport in VIEWPORTS
    ]
).sort_values("area_px2")
display(viewport_table)

In [ ]:
def convolution_scores(saliency_maps: torch.Tensor, viewport: Viewport) -> torch.Tensor:
    """Return raw saliency mass at every valid top-left corner: [T, Hout, Wout]."""
    if saliency_maps.ndim != 3:
        raise ValueError(f"Expected [T,H,W], received {tuple(saliency_maps.shape)}")
    maps = saliency_maps.to(DEVICE, non_blocking=True).float().unsqueeze(1)
    kernel = torch.ones((1, 1, viewport.height, viewport.width), dtype=maps.dtype, device=DEVICE)
    return F.conv2d(maps, kernel).squeeze(1)


def candidate_peaks(saliency_maps: torch.Tensor, viewports: Iterable[Viewport]) -> Dict[str, Dict[str, torch.Tensor]]:
    saliency_maps = saliency_maps.to(DEVICE, non_blocking=True)
    total_mass = saliency_maps.sum(dim=(1, 2)).clamp_min(torch.finfo(saliency_maps.dtype).eps)
    output: Dict[str, Dict[str, torch.Tensor]] = {}
    for viewport in viewports:
        scores = convolution_scores(saliency_maps, viewport)
        flattened = scores.flatten(1)
        mass, flat_index = flattened.max(dim=1)
        output[viewport.ratio] = {
            "scores": scores,
            "peak_mass": mass,
            "coverage": mass / total_mass,
            "y": flat_index // scores.shape[-1],
            "x": flat_index % scores.shape[-1],
        }
    return output


def choose_independent(peaks: Dict[str, Dict[str, torch.Tensor]], viewports: Iterable[Viewport], threshold: float) -> List[dict]:
    ordered = sorted(viewports, key=lambda viewport: (viewport.area, viewport.ratio))
    frames = next(iter(peaks.values()))["peak_mass"].shape[0]
    decisions = []
    for frame in range(frames):
        best_mass = max(float(peaks[viewport.ratio]["peak_mass"][frame]) for viewport in ordered)
        eligible = [
            viewport
            for viewport in ordered
            if float(peaks[viewport.ratio]["peak_mass"][frame]) >= threshold * best_mass
        ]
        viewport = eligible[0]
        candidate = peaks[viewport.ratio]
        decisions.append(
            {
                "frame": frame,
                "ratio": viewport.ratio,
                "y": int(candidate["y"][frame]),
                "x": int(candidate["x"][frame]),
                "height": viewport.height,
                "width": viewport.width,
                "area": viewport.area,
                "peak_mass": float(candidate["peak_mass"][frame]),
                "coverage": float(candidate["coverage"][frame]),
                "best_mass": best_mass,
            }
        )
    return decisions


peaks0 = candidate_peaks(maps0, VIEWPORTS)
assert all(value["scores"].is_cuda for value in peaks0.values())
independent0 = choose_independent(peaks0, VIEWPORTS, COVERAGE_THRESHOLD)
independent_table0 = pd.DataFrame(independent0)
assert (independent_table0["peak_mass"] >= COVERAGE_THRESHOLD * independent_table0["best_mass"] - 1e-6).all()
display(independent_table0.head())

In [ ]:
def temporal_states(
    saliency_maps: torch.Tensor,
    peaks: Dict[str, Dict[str, torch.Tensor]],
    viewports: Iterable[Viewport],
    threshold: float,
) -> List[dict]:
    """Keep every GPU placement that meets the same 90%-of-best constraint as the baseline."""
    saliency_maps = saliency_maps.to(DEVICE, non_blocking=True)
    total_mass = saliency_maps.sum(dim=(1, 2)).clamp_min(torch.finfo(saliency_maps.dtype).eps)
    frame_height, frame_width = saliency_maps.shape[-2:]
    frame_area = frame_height * frame_width
    viewports = list(viewports)
    ratio_ids = {viewport.ratio: index for index, viewport in enumerate(viewports)}
    states_by_frame: List[dict] = []
    for frame in range(saliency_maps.shape[0]):
        best_mass = torch.stack([peaks[viewport.ratio]["peak_mass"][frame] for viewport in viewports]).max()
        pieces = []
        for viewport in viewports:
            score = peaks[viewport.ratio]["scores"][frame]
            yy, xx = torch.where(score >= threshold * best_mass)
            if yy.numel() == 0:
                continue
            mass = score[yy, xx]
            count = yy.shape[0]
            pieces.append(
                {
                    "ratio_id": torch.full((count,), ratio_ids[viewport.ratio], dtype=torch.long, device=DEVICE),
                    "y": yy,
                    "x": xx,
                    "height": torch.full((count,), viewport.height, dtype=torch.long, device=DEVICE),
                    "width": torch.full((count,), viewport.width, dtype=torch.long, device=DEVICE),
                    "area": torch.full((count,), viewport.area, dtype=torch.long, device=DEVICE),
                    "peak_mass": mass,
                    "center": torch.stack(
                        ((yy + viewport.height / 2) / frame_height, (xx + viewport.width / 2) / frame_width),
                        dim=1,
                    ),
                    "local_cost": viewport.area / frame_area - 1e-3 * (mass / total_mass[frame]),
                }
            )
        if not pieces:
            raise RuntimeError(f"No feasible temporal state at frame {frame}")
        states_by_frame.append(
            {
                key: torch.cat([piece[key] for piece in pieces])
                for key in pieces[0]
            }
            | {"best_mass": best_mass, "total_mass": total_mass[frame]}
        )
        assert states_by_frame[-1]["local_cost"].is_cuda
    return states_by_frame


def temporal_dynamic_programming(states_by_frame: List[dict], center_penalty: float, aspect_penalty: float) -> List[dict]:
    costs = states_by_frame[0]["local_cost"]
    backpointers: List[torch.Tensor] = []
    for frame in range(1, len(states_by_frame)):
        previous, current = states_by_frame[frame - 1], states_by_frame[frame]
        displacement = torch.linalg.norm(current["center"][:, None, :] - previous["center"][None, :, :], dim=2)
        aspect_switch = (current["ratio_id"][:, None] != previous["ratio_id"][None, :]).to(costs.dtype)
        total_cost = center_penalty * displacement + aspect_penalty * aspect_switch + costs[None, :]
        predecessor = total_cost.argmin(dim=1)
        costs = current["local_cost"] + total_cost[torch.arange(predecessor.shape[0], device=DEVICE), predecessor]
        backpointers.append(predecessor)

    chosen = [int(costs.argmin())]
    for predecessor in reversed(backpointers):
        chosen.append(int(predecessor[chosen[-1]]))
    chosen.reverse()
    ratio_names = list(ASPECT_RATIOS)
    decisions = []
    for frame, state_index in enumerate(chosen):
        state = states_by_frame[frame]
        mass = float(state["peak_mass"][state_index])
        total = float(state["total_mass"])
        decisions.append(
            {
                "frame": frame,
                "ratio": ratio_names[int(state["ratio_id"][state_index])],
                "y": int(state["y"][state_index]),
                "x": int(state["x"][state_index]),
                "height": int(state["height"][state_index]),
                "width": int(state["width"][state_index]),
                "area": int(state["area"][state_index]),
                "peak_mass": mass,
                "coverage": mass / total,
                "best_mass": float(state["best_mass"]),
                "center_y": float(state["center"][state_index, 0]),
                "center_x": float(state["center"][state_index, 1]),
                "local_cost": float(state["local_cost"][state_index]),
            }
        )
    return decisions


states0 = temporal_states(maps0, peaks0, VIEWPORTS, COVERAGE_THRESHOLD)
assert all(state["local_cost"].is_cuda and state["center"].is_cuda for state in states0)
temporal0 = temporal_dynamic_programming(states0, CENTER_VELOCITY_PENALTY, ASPECT_SWITCH_PENALTY)
temporal_table0 = pd.DataFrame(temporal0)
assert len(temporal_table0) == maps0.shape[0]
assert ((temporal_table0["x"] >= 0) & (temporal_table0["y"] >= 0)).all()
assert ((temporal_table0["x"] + temporal_table0["width"] <= maps0.shape[-1])).all()
assert ((temporal_table0["y"] + temporal_table0["height"] <= maps0.shape[-2])).all()
display(temporal_table0.head())

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)
COLORS = {"independent": (255, 190, 0), "temporal": (0, 230, 255)}  # RGB


def denormalize_frame(frame: torch.Tensor) -> np.ndarray:
    image = frame.detach().cpu().numpy() * IMAGENET_STD + IMAGENET_MEAN
    return np.transpose(np.clip(image, 0, 1), (1, 2, 0))


def saliency_overlay(frame: torch.Tensor, saliency: torch.Tensor) -> np.ndarray:
    image = denormalize_frame(frame)
    sal = saliency.detach().cpu().numpy().astype(np.float32)
    sal = (sal - sal.min()) / max(float(sal.max() - sal.min()), 1e-8)
    heat = cv2.cvtColor(cv2.applyColorMap(np.uint8(np.round(sal * 255)), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB) / 255.0
    return np.clip(0.55 * image + 0.45 * heat, 0, 1)


def draw_viewport(image: np.ndarray, decision: dict, color: Tuple[int, int, int], label: str) -> np.ndarray:
    canvas = np.uint8(np.clip(image * 255, 0, 255)).copy()
    x0, y0 = int(decision["x"]), int(decision["y"])
    x1, y1 = x0 + int(decision["width"]), y0 + int(decision["height"])
    cv2.rectangle(canvas, (x0, y0), (x1 - 1, y1 - 1), color, 2)
    cv2.putText(canvas, label, (x0 + 4, min(y0 + 18, canvas.shape[0] - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
    return canvas / 255.0


def side_by_side_frame(clip: torch.Tensor, maps: torch.Tensor, independent: List[dict], temporal: List[dict], frame: int) -> np.ndarray:
    base = saliency_overlay(clip[frame], maps[frame])
    baseline = draw_viewport(base, independent[frame], COLORS["independent"], "independent")
    smoothed = draw_viewport(base, temporal[frame], COLORS["temporal"], "temporal")
    combined = smoothed.copy()
    combined = draw_viewport(combined, independent[frame], COLORS["independent"], "independent")
    return np.concatenate([baseline, smoothed, combined], axis=1)


def plot_contact_sheet(clip: torch.Tensor, maps: torch.Tensor, independent: List[dict], temporal: List[dict], title: str) -> None:
    frames = [0, 7, 15, 23, 31]
    fig, axes = plt.subplots(len(frames), 3, figsize=(16, 3.4 * len(frames)))
    for row, frame in enumerate(frames):
        rendered = side_by_side_frame(clip, maps, independent, temporal, frame)
        width = rendered.shape[1] // 3
        for col, name in enumerate(("Independent", "Temporal DP", "Both")):
            axes[row, col].imshow(rendered[:, col * width : (col + 1) * width])
            axes[row, col].set_title(f"Frame {frame + 1} — {name}")
            axes[row, col].axis("off")
    fig.suptitle(title, y=1.0)
    fig.tight_layout()


plot_contact_sheet(clip0, maps0, independent0, temporal0, "Clip 0: independent vs temporal viewport paths")

In [ ]:
REVIEW_FRAME = 15

fig, axes = plt.subplots(1, len(VIEWPORTS) + 1, figsize=(18, 4))
axes[0].imshow(saliency_overlay(clip0[REVIEW_FRAME], maps0[REVIEW_FRAME]))
axes[0].set_title(f"Frame {REVIEW_FRAME + 1} saliency")
axes[0].axis("off")

for axis, viewport in zip(axes[1:], VIEWPORTS):
    score = peaks0[viewport.ratio]["scores"][REVIEW_FRAME].detach().cpu().numpy()
    axis.imshow(score, cmap="magma", origin="upper", aspect="auto")
    y, x = np.unravel_index(score.argmax(), score.shape)
    axis.scatter([x], [y], color="cyan", s=32)
    coverage = peaks0[viewport.ratio]["coverage"][REVIEW_FRAME].item()
    axis.set_title(f"{viewport.ratio} score map\npeak coverage={coverage:.1%}")
    axis.set_xlabel("top-left x (px)")
    axis.set_ylabel("top-left y (px)")
fig.suptitle("Convolutional saliency-mass score maps — selected peak shown in cyan", y=1.05)
fig.tight_layout()

In [ ]:
def animate_clip(clip: torch.Tensor, maps: torch.Tensor, independent: List[dict], temporal: List[dict]):
    figure, axis = plt.subplots(figsize=(14, 4.3))
    axis.axis("off")
    artist = axis.imshow(side_by_side_frame(clip, maps, independent, temporal, 0))

    def update(frame: int):
        artist.set_data(side_by_side_frame(clip, maps, independent, temporal, frame))
        axis.set_title(
            f"Frame {frame + 1}/32 | independent={independent[frame]['ratio']} | temporal={temporal[frame]['ratio']}"
        )
        return (artist,)

    animation = FuncAnimation(figure, update, frames=range(clip.shape[0]), interval=250, blit=True)
    plt.close(figure)
    return animation


animation0 = animate_clip(clip0, maps0, independent0, temporal0)
HTML(animation0.to_jshtml())

In [ ]:
def center_displacement(decisions: List[dict]) -> np.ndarray:
    centers = np.array(
        [
            ((decision["y"] + decision["height"] / 2) / 224, (decision["x"] + decision["width"] / 2) / 384)
            for decision in decisions
        ]
    )
    return np.r_[0.0, np.linalg.norm(np.diff(centers, axis=0), axis=1)]


def evaluate_clip(dataset_index: int) -> Tuple[torch.Tensor, torch.Tensor, List[dict], List[dict]]:
    clip, _last_map, maps = dataset[dataset_index]
    peaks = candidate_peaks(maps, VIEWPORTS)
    independent = choose_independent(peaks, VIEWPORTS, COVERAGE_THRESHOLD)
    temporal = temporal_dynamic_programming(
        temporal_states(maps, peaks, VIEWPORTS, COVERAGE_THRESHOLD),
        CENTER_VELOCITY_PENALTY,
        ASPECT_SWITCH_PENALTY,
    )
    return clip, maps, independent, temporal


all_rows = []
clip_results = {}
for dataset_index in manifest_indexes:
    clip, maps, independent, temporal = evaluate_clip(dataset_index)
    clip_results[dataset_index] = (clip, maps, independent, temporal)
    for method, decisions in (("independent", independent), ("temporal", temporal)):
        velocity = center_displacement(decisions)
        for decision, displacement in zip(decisions, velocity):
            all_rows.append(
                {
                    "clip_index": dataset_index,
                    "method": method,
                    "frame": decision["frame"] + 1,
                    "ratio": decision["ratio"],
                    "coverage": decision["coverage"],
                    "area_fraction": decision["area"] / (224 * 384),
                    "center_displacement": displacement,
                    "x": decision["x"],
                    "y": decision["y"],
                }
            )

results = pd.DataFrame(all_rows)
summary = (
    results.groupby("method")
    .agg(
        mean_coverage=("coverage", "mean"),
        mean_area_fraction=("area_fraction", "mean"),
        mean_center_displacement=("center_displacement", "mean"),
    )
    .reset_index()
)
display(summary.style.format({"mean_coverage": "{:.1%}", "mean_area_fraction": "{:.1%}", "mean_center_displacement": "{:.3f}"}))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

ratio_counts = results.groupby(["method", "ratio"]).size().unstack(fill_value=0)
ratio_counts.plot(kind="bar", ax=axes[0])
axes[0].set_title("Selected viewport ratio count")
axes[0].set_xlabel("Selection method")
axes[0].set_ylabel("Frames (count)")
axes[0].legend(title="Aspect ratio")

for method, group in results.groupby("method"):
    axes[1].hist(group["coverage"], bins=20, alpha=0.6, label=method)
axes[1].set_title("Viewport saliency coverage distribution")
axes[1].set_xlabel("Selected viewport / full-frame saliency mass")
axes[1].set_ylabel("Frames (count)")
axes[1].legend(title="Method")

for method, group in results.groupby("method"):
    axes[2].plot(group["center_displacement"].to_numpy(), alpha=0.35, label=method)
axes[2].set_title("Viewport center displacement by frame")
axes[2].set_xlabel("Evaluation row (clip-major frame index)")
axes[2].set_ylabel("Normalized center displacement")
axes[2].legend(title="Method")

fig.suptitle("Eight training clips × 32 frames — convolutional viewport-selection ablation", y=1.04)
fig.tight_layout()

coverage_tradeoff = results.groupby("method").agg(coverage=("coverage", "mean"), area=("area_fraction", "mean"), velocity=("center_displacement", "mean"))
display(coverage_tradeoff.style.format("{:.3f}"))